In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from src.features.build_features import build_features
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV

In [24]:
df = pd.read_csv('../Hotel Reservations.csv')

In [25]:
df = build_features(df)

New number of features: 27


In [26]:
y = df['booking_status'].map({'Canceled': 1, 'Not_Canceled': 0})
X = df.drop(columns = ["Booking_ID", "booking_status", 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [27]:
cat_cols = ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type', 'arrival_season']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [28]:
encoder.fit(X_train[cat_cols])

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [29]:
def apply_encoding(data_subset, encoder, columns):
    encoded = encoder.transform(data_subset[columns])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(columns), index=data_subset.index)
    return pd.concat([data_subset.drop(columns, axis=1), encoded_df], axis=1)

In [30]:
X_train_encoded = apply_encoding(X_train, encoder, cat_cols)
X_test_encoded = apply_encoding(X_test, encoder, cat_cols)

In [31]:
numerical_cols = ['lead_time', 'avg_price_per_room', 'total_people', 'log_lead_time']
scaler = RobustScaler()

In [32]:
scaler.fit(X_train_encoded[numerical_cols])
X_train_final = X_train_encoded.copy()
X_test_final = X_test_encoded.copy()

X_train_final[numerical_cols] = scaler.transform(X_train_encoded[numerical_cols])
X_test_final[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

In [33]:
################################
# Linear Discriminant Analysis #
################################

In [34]:
lda = LinearDiscriminantAnalysis()

param_grid_lda = [
    {
        'solver': ['svd']
    },
    {
        'solver': ['lsqr', 'eigen'],
        'shrinkage': [None, 'auto', 0.01, 0.1, 0.3, 0.5, 0.7, 1.0]
    }
]

In [35]:
grid_search_lda = GridSearchCV(
    estimator=lda,
    param_grid=param_grid_lda,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [36]:
grid_search_lda.fit(X_train_final, y_train)

Fitting 5 folds for each of 17 candidates, totalling 85 fits


c:\Users\Szef\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
5 fits failed out of a total of 85.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Szef\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Szef\anaconda3\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Szef\anaconda3\Lib\site-packages\sklearn\discriminant_analysis.py", line 637, in fit
    self._solve_eigen(

GridSearchCV(cv=5, estimator=LinearDiscriminantAnalysis(), n_jobs=-1,
             param_grid=[{'solver': ['svd']},
                         {'shrinkage': [None, 'auto', 0.01, 0.1, 0.3, 0.5, 0.7,
                                        1.0],
                          'solver': ['lsqr', 'eigen']}],
             scoring='f1', verbose=1)

In [37]:
print(f"Best parameters: {grid_search_lda.best_params_}")
print(f"Best F1 score during CV: {grid_search_lda.best_score_:.4f}")

best_lda_model = grid_search_lda.best_estimator_


test_preds = best_lda_model.predict(X_test_final)

print(classification_report(y_test, test_preds))

Best parameters: {'shrinkage': 'auto', 'solver': 'lsqr'}
Best F1 score during CV: 0.6833
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      4878
           1       0.76      0.65      0.70      2377

    accuracy                           0.82      7255
   macro avg       0.80      0.77      0.78      7255
weighted avg       0.81      0.82      0.81      7255



In [38]:
###################################
# Quadratic Discriminant Analysis #
###################################

In [39]:
qda = QuadraticDiscriminantAnalysis()

param_grid_qda = {
    'reg_param': [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
}

In [40]:
grid_search_qda = GridSearchCV(
    estimator=qda,
    param_grid=param_grid_qda,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

In [41]:
grid_search_qda.fit(X_train_final, y_train)

c:\Users\Szef\anaconda3\Lib\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


GridSearchCV(cv=5, estimator=QuadraticDiscriminantAnalysis(), n_jobs=-1,
             param_grid={'reg_param': [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]},
             scoring='f1')

In [42]:
print(f"Best parameters: {grid_search_qda.best_params_}")
print(f"Best F1 score during CV: {grid_search_qda.best_score_:.4f}")

best_lda_model = grid_search_qda.best_estimator_


test_preds = best_lda_model.predict(X_test_final)

print(classification_report(y_test, test_preds))

Best parameters: {'reg_param': 0.5}
Best F1 score during CV: 0.6716
              precision    recall  f1-score   support

           0       0.88      0.74      0.80      4878
           1       0.60      0.78      0.68      2377

    accuracy                           0.76      7255
   macro avg       0.74      0.76      0.74      7255
weighted avg       0.78      0.76      0.76      7255

